# Day 8 - SVM Classifier

Classifying tumors as benign or malignant using Support Vector Machine. The key idea of SVM is finding the hyperplane that separates classes with maximum margin.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

%matplotlib inline

## 1. Load Data

In [ ]:
df = pd.read_csv('dataset/cancer.csv')
print('Shape:', df.shape)
print('\nClass distribution:')
print(df['diagnosis'].value_counts())
df.head()

## 2. EDA

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 9))
for ax, feature in zip(axes.flatten(), ['radius', 'concavity', 'compactness', 'texture']):
    ax.hist(df[df['diagnosis']==0][feature], alpha=0.6, label='Benign', color='#4CAF50', bins=25)
    ax.hist(df[df['diagnosis']==1][feature], alpha=0.6, label='Malignant', color='#F44336', bins=25)
    ax.set_title(feature)
    ax.legend(fontsize=8)
plt.suptitle('Feature Distributions by Diagnosis')
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(9, 7))
sns.heatmap(df.corr(), annot=True, fmt='.2f', cmap='coolwarm', linewidths=0.5)
plt.title('Feature Correlation')
plt.show()

## 3. Train / Test Split

In [ ]:
feature_cols = ['radius', 'texture', 'perimeter', 'area', 'smoothness',
                'compactness', 'concavity', 'symmetry', 'fractal_dimension']

X = df[feature_cols]
y = df['diagnosis']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# SVM is sensitive to feature scale - always scale
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

## 4. Compare Kernels

In [ ]:
for kernel in ['linear', 'rbf', 'poly']:
    svm = SVC(kernel=kernel, random_state=42)
    scores = cross_val_score(svm, X_train_scaled, y_train, cv=5)
    print(f'{kernel}: {scores.mean():.4f} +/- {scores.std():.4f}')

## 5. Train Best Model

In [ ]:
model = SVC(kernel='rbf', C=1.0, gamma='scale', probability=True, random_state=42)
model.fit(X_train_scaled, y_train)

preds = model.predict(X_test_scaled)
print(f'Test Accuracy: {accuracy_score(y_test, preds):.4f}\n')
print(classification_report(y_test, preds, target_names=['Benign', 'Malignant']))

In [ ]:
cm = confusion_matrix(y_test, preds)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Benign', 'Malignant'],
            yticklabels=['Benign', 'Malignant'])
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.title('Confusion Matrix - SVM RBF')
plt.show()

## Wrap-up

SVM with RBF kernel achieved 88% accuracy. The linear kernel actually did slightly better (89%) which suggests the data is roughly linearly separable.

Key takeaway: SVM must be scaled before training — it is one of the most scale-sensitive algorithms. Without scaling, performance drops significantly.